# Notebook 06 — PDF Figure Extraction & Captioning

**Phase 4 · Task group 116.** Goal: unpack the CORD-19 PDFs subset into `(figure.png, caption)` pairs, fill in missing captions with BLIP-2, and produce `figures_metadata.parquet` — the single input to Notebook 07's CLIP embedder.

### Why this phase exists
Text-only RAG discards exactly the content scientists *cite* when reviewing a paper — figures and tables. The RQ1 answer depends on us recovering that signal cleanly.

### Pipeline
1. Discover PDFs under `1_data/raw/pdfs/` (Tier 1 = 50–100 papers is enough for local debugging; Tier 2 pulls the Kaggle PDF subset).
2. Use PyMuPDF to dump embedded images per page → `1_data/figures/<tier>/<cord_uid>_fig_<k>.png`.
3. Use PyMuPDF page text + heuristics ("Fig. N", "Figure N.") to grab the caption nearest each figure.
4. For figures whose caption is empty or junk, run **BLIP-2 (Salesforce/blip2-opt-2.7b)** to generate a caption.
5. Emit `figures_metadata.parquet` with one row per figure.

### Local vs Kaggle
On a laptop BLIP-2 is painful; we default `RUN_BLIP2=False` for Tier 1 so the notebook finishes in minutes. Flip it to `True` for Kaggle Tier 2 to fill in the caption gaps.


In [2]:
import os, sys, json, re, time, warnings
from pathlib import Path
from typing import List, Dict

NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if (NB_DIR.parent / "2_src").exists() else NB_DIR
sys.path.insert(0, str(PROJECT_ROOT / "2_src"))

os.environ.setdefault("SCIRET_TIER", "tier1")
from config import get_config, BLIP2_MODEL, SEED
CFG = get_config()
print(CFG.summary())

import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# Toggle: set to True on Kaggle Tier 2 where GPU is available.
RUN_BLIP2 = os.environ.get("SCIRET_BLIP2", "0") == "1"
print("RUN_BLIP2:", RUN_BLIP2)


[SciRet:tier1] size=1000 root=D:\SciRet-Scientific-Information-Made-Easy\Sciret2 chunks=chunks.parquet chroma=chroma_db/sciret_tier1_bge_m3_cs400_o50
RUN_BLIP2: False


## 1. Discover PDFs

In [3]:
import fitz  # PyMuPDF

PDF_ROOT = CFG.pdf_dir
PDF_ROOT.mkdir(parents=True, exist_ok=True)
pdf_paths = sorted(PDF_ROOT.rglob("*.pdf"))
print(f"PDFs found under {PDF_ROOT}: {len(pdf_paths)}")
if not pdf_paths:
    print(
        "No PDFs yet. Download a subset (50-100 papers for Tier 1) from "
        "https://www.kaggle.com/datasets/allen-institute-for-ai/CORD-19-research-challenge "
        "and drop them into 1_data/raw/pdfs/. "
        "The rest of this notebook is a no-op until that is done."
    )


PDFs found under D:\SciRet-Scientific-Information-Made-Easy\Sciret2\1_data\raw\pdfs: 0
No PDFs yet. Download a subset (50-100 papers for Tier 1) from https://www.kaggle.com/datasets/allen-institute-for-ai/CORD-19-research-challenge and drop them into 1_data/raw/pdfs/. The rest of this notebook is a no-op until that is done.


## 2. Figure + caption extraction

In [4]:
CAPTION_RE = re.compile(
    r"(fig(?:ure)?\.?\s*\d+[a-z]?\s*[:.\-]?\s*.{0,400})",
    flags=re.IGNORECASE | re.DOTALL,
)

def _nearest_caption(page_text: str, img_idx: int) -> str:
    """Return the first Figure/Fig caption in this page; good enough for
    per-page single-figure papers. For multi-figure pages we later fall back
    to BLIP-2 for figures beyond the first."""
    matches = CAPTION_RE.findall(page_text or "")
    if not matches:
        return ""
    m = matches[min(img_idx, len(matches)-1)]
    return " ".join(m.split())[:500]

def extract_from_pdf(pdf_path: Path, out_dir: Path) -> List[Dict]:
    records = []
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        print("  ! failed to open", pdf_path.name, ":", e)
        return records

    cord_uid = pdf_path.stem
    fig_counter = 0
    out_dir.mkdir(parents=True, exist_ok=True)

    for page_idx, page in enumerate(doc):
        page_text = page.get_text("text")
        images = page.get_images(full=True)
        for img_idx_on_page, img in enumerate(images):
            xref = img[0]
            try:
                pix = fitz.Pixmap(doc, xref)
                if pix.n > 4:  # CMYK -> RGB
                    pix = fitz.Pixmap(fitz.csRGB, pix)
                if pix.width < 100 or pix.height < 100:
                    pix = None; continue  # skip logos / tiny icons
                fid = f"{cord_uid}_fig_{fig_counter:03d}"
                asset = out_dir / f"{fid}.png"
                pix.save(str(asset))
                records.append({
                    "figure_id": fid,
                    "cord_uid": cord_uid,
                    "page": page_idx,
                    "img_idx_on_page": img_idx_on_page,
                    "asset_path": str(asset),
                    "caption_raw": _nearest_caption(page_text, img_idx_on_page),
                    "width": pix.width,
                    "height": pix.height,
                })
                fig_counter += 1
            except Exception as e:
                print(f"  ! image extract failed on {pdf_path.name} p{page_idx}: {e}")
                continue
            finally:
                pix = None
    return records

figure_records: List[Dict] = []
for i, pdf in enumerate(pdf_paths):
    figure_records.extend(extract_from_pdf(pdf, CFG.figures_dir))
    if (i+1) % 10 == 0:
        print(f"  processed {i+1}/{len(pdf_paths)} PDFs  -> {len(figure_records)} figures")

print("total figures:", len(figure_records))


total figures: 0


In [5]:
df_fig = pd.DataFrame(figure_records) if figure_records else pd.DataFrame(
    columns=["figure_id","cord_uid","page","img_idx_on_page","asset_path",
             "caption_raw","width","height"]
)
df_fig.head(6)


,figure_id,cord_uid,page,img_idx_on_page,asset_path,caption_raw,width,height


## 3. Caption coverage

In [6]:
if len(df_fig):
    cov = (df_fig["caption_raw"].str.len() > 30).mean() * 100
    print(f"figures with >30-char caption: {cov:.1f}%")
    # sample captions
    print("\n-- examples --")
    for _, r in df_fig.sample(min(5,len(df_fig)), random_state=SEED).iterrows():
        print(f"[{r.figure_id}] {r.caption_raw[:200]!r}")


## 4. BLIP-2 fallback captioning

Runs only if `RUN_BLIP2=True` (Kaggle) or if you explicitly opt in. We cap it at 200 figures per session to keep runtime sane.

In [7]:
df_fig["caption"] = df_fig["caption_raw"].fillna("").str.strip()

def need_blip(row):
    return len(row["caption"]) < 30

if RUN_BLIP2 and len(df_fig):
    import torch
    from PIL import Image
    from transformers import Blip2Processor, Blip2ForConditionalGeneration

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("loading BLIP-2 on", DEVICE)
    proc = Blip2Processor.from_pretrained(BLIP2_MODEL)
    blip = Blip2ForConditionalGeneration.from_pretrained(
        BLIP2_MODEL,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    ).to(DEVICE).eval()

    todo_idx = df_fig.index[df_fig.apply(need_blip, axis=1)].tolist()
    todo_idx = todo_idx[:200]
    print(f"captioning {len(todo_idx)} figures with BLIP-2 …")

    captions = []
    for i, idx in enumerate(todo_idx):
        try:
            img = Image.open(df_fig.loc[idx, "asset_path"]).convert("RGB")
            inputs = proc(images=img, return_tensors="pt").to(DEVICE,
                torch.float16 if DEVICE == "cuda" else torch.float32)
            out = blip.generate(**inputs, max_new_tokens=40)
            cap = proc.decode(out[0], skip_special_tokens=True).strip()
        except Exception as e:
            cap = ""
            print(f"  ! BLIP failed on {df_fig.loc[idx,'figure_id']}: {e}")
        captions.append((idx, cap))
        if (i+1) % 20 == 0:
            print(f"    {i+1}/{len(todo_idx)}")

    for idx, cap in captions:
        if cap:
            df_fig.at[idx, "caption"] = f"[blip2] {cap}"
else:
    print("Skipping BLIP-2 (RUN_BLIP2=False).  caption column = caption_raw as-is.")


Skipping BLIP-2 (RUN_BLIP2=False).  caption column = caption_raw as-is.


## 5. Compose `text_for_embedding`

Notebook 07 embeds figures with CLIP (image side) *and* embeds their captions with BGE-M3 (text side). The `text_for_embedding` column is the canonical caption string we hand to the text branch.

In [8]:
def compose(row):
    parts = [row.get("caption","") or ""]
    # we may later bolt on nearby paragraph text here at Tier 2.
    return " ".join([p for p in parts if p]).strip()[:500]

if len(df_fig):
    df_fig["text_for_embedding"] = df_fig.apply(compose, axis=1)

out = CFG.figures_metadata_path
df_fig.to_parquet(out, index=False)
print(f"saved -> {out}  rows={len(df_fig)}")

# quick visual — preview a handful of extracted figures
if len(df_fig):
    import matplotlib.image as mpimg
    show = df_fig.sample(min(6, len(df_fig)), random_state=SEED)
    fig, axes = plt.subplots(2, 3, figsize=(11, 6))
    for ax, (_, r) in zip(axes.flatten(), show.iterrows()):
        try:
            ax.imshow(mpimg.imread(r["asset_path"]))
        except Exception:
            pass
        ax.set_title(r["figure_id"], fontsize=8)
        ax.axis("off")
    fig.suptitle("Random extracted figures")
    fig.tight_layout()
    fig.savefig(CFG.results_dir / "figure_extraction_examples.png", dpi=120)
    plt.show()


saved -> D:\SciRet-Scientific-Information-Made-Easy\Sciret2\1_data\processed\tier1\figures_metadata.parquet  rows=0


---
**Outputs**
* `1_data/figures/<tier>/*.png` — extracted figure assets
* `1_data/processed/<tier>/figures_metadata.parquet` — one row per figure
* `4_results/<tier>/figure_extraction_examples.png`

**Next:** Notebook 07 — CLIP embeddings over the figures we just extracted.
